In [1]:
import pandas as pd

In [2]:
deliveries = pd.read_csv("../datasets/original_deliveries.csv")

# Identify wickets
deliveries["is_wicket"] = deliveries["dismissal_kind"].notna().astype(int)

# Aggregate bowler performance per match WITH TEAM
bowler_stats = (
    deliveries
    .groupby(["match_id", "bowler", "bowling_team"])
    .agg(
        wickets=("is_wicket", "sum"),
        balls=("ball", "count"),
        runs_conceded=("total_runs", "sum")
    )
    .reset_index()
)

# Economy rate
bowler_stats["economy"] = bowler_stats["runs_conceded"] / (bowler_stats["balls"] / 6)

# Sort for rolling calculations
bowler_stats = bowler_stats.sort_values(["bowler", "match_id"])

# Rolling features
bowler_stats["wkts_last5"] = (
    bowler_stats.groupby("bowler")["wickets"]
    .rolling(5).mean()
    .reset_index(level=0, drop=True)
)

bowler_stats["eco_last5"] = (
    bowler_stats.groupby("bowler")["economy"]
    .rolling(5).mean()
    .reset_index(level=0, drop=True)
)

# Drop rows without full history
final_wickets_df = bowler_stats.dropna()

# Rename columns
final_wickets_df.rename(columns={
    "wickets": "next_match_wkts",
    "bowling_team": "team"
}, inplace=True)

# Save dataset
final_wickets_df.to_csv("../datasets/wicket_dataset.csv", index=False)

print("wicket_dataset.csv created successfully with TEAM column")
   


wicket_dataset.csv created successfully with TEAM column


C:\Users\JASMIN\AppData\Local\Temp\ipykernel_2492\3812541336.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_wickets_df.rename(columns={


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import joblib
import pandas as pd

# --------------------------------------------------
# Load wicket dataset
# --------------------------------------------------
df = pd.read_csv("../datasets/wicket_dataset.csv")

# --------------------------------------------------
# Features & Target (ONLY EXISTING COLUMNS)
# --------------------------------------------------
X = df[
    [
        "wkts_last5",
        "eco_last5",
        "balls",
        "economy"
    ]
]

y = df["next_match_wkts"]

# --------------------------------------------------
# ML Pipeline
# --------------------------------------------------
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        random_state=42
    ))
])

# --------------------------------------------------
# Train model
# --------------------------------------------------
pipeline.fit(X, y)

# --------------------------------------------------
# Save trained model
# --------------------------------------------------
joblib.dump(pipeline, "../wickets_pipeline.joblib")

print("✅ Wicket prediction model trained and saved successfully.")







✅ Wicket prediction model trained and saved successfully.
